In [ ]:
# ══════════════════════════════════════════════════════════════════
# Colab setup — self-contained. No git clone, no FORK_URL, no sys.path.
# Before running: Runtime → Change runtime type → T4 GPU (free tier)
# ══════════════════════════════════════════════════════════════════
!pip install -q kailash polars plotly gdown python-dotenv nest-asyncio httpx kailash-ml

import nest_asyncio; nest_asyncio.apply()

import torch
if torch.cuda.is_available():
    print(f'✓ GPU available: {torch.cuda.get_device_name(0)}')
else:
    print('⚠ No GPU — training will be slow. Runtime → Change runtime type → T4 GPU')

print('✓ Setup complete. All helpers defined in the next cell.')


In [ ]:
# ══════════════════════════════════════════════════════════════════
# MLFP inlined helpers — DO NOT EDIT (collapse this cell!)
# Auto-generated by scripts/generate_selfcontained_notebook.py for mlfp02
# ══════════════════════════════════════════════════════════════════
from __future__ import annotations

# ── shared/kailash_helpers.py ──
"""Common Kailash SDK setup patterns for MLFP exercises."""


import logging
import os
from dataclasses import dataclass, field, replace
from pathlib import Path
from typing import Any, Literal

import numpy as np
import polars as pl
from dotenv import load_dotenv

logger = logging.getLogger(__name__)


def setup_environment() -> None:
    """Load .env and validate common configuration.

    Call this at the top of every exercise that needs API keys or DB connections.
    """
    # Find .env by walking up from the exercise file
    env_path = Path.cwd() / ".env"
    if not env_path.exists():
        # Try repo root
        for parent in Path.cwd().parents:
            candidate = parent / ".env"
            if candidate.exists():
                env_path = candidate
                break

    load_dotenv(env_path)


def get_connection_manager(db_url: str | None = None):
    """Create a ConnectionManager for kailash-ml engines.

    Args:
        db_url: Database URL. Defaults to SQLite at ./mlfp.db
    """
    from kailash.db import ConnectionManager

    url = db_url or os.environ.get("DATABASE_URL", "sqlite:///mlfp.db")
    return ConnectionManager(url)


def get_device() -> "torch.device":
    """Return the best available compute device as a ``torch.device``.

    Routes through ``kailash_ml.device()`` (the canonical kailash-ml 0.12+
    accelerator detector) so MPS / CUDA / ROCm / Intel XPU / CPU are all
    selected by the same logic the rest of the platform uses. The previous
    hand-rolled ``mps→cuda→cpu`` cascade is replaced because:

      * kailash-ml's detector knows about ROCm, Intel XPU, and fp16/bf16
        capability flags — the cascade in this helper did not.
      * Apple-Silicon students get the Metal Performance Shaders backend
        with mixed-precision (fp16) without any opt-in.
      * One detection point means lessons that print "Using device: …"
        agree with what kailash-ml's MLEngine() actually picks.
    """
    import kailash_ml as km
    import torch

    backend = km.device()  # BackendInfo (auto MPS on Mac, CUDA on Linux+NVIDIA, …)
    return torch.device(backend.device_string)


def get_llm_model() -> str:
    """Get the configured LLM model name from environment."""
    setup_environment()
    model = os.environ.get("DEFAULT_LLM_MODEL", os.environ.get("OPENAI_PROD_MODEL"))
    if not model:
        raise EnvironmentError(
            "No LLM model configured. Set DEFAULT_LLM_MODEL or OPENAI_PROD_MODEL in .env"
        )
    return model


# ════════════════════════════════════════════════════════════════════════
# SPLIT FIRST, THEN PREPROCESS
# ════════════════════════════════════════════════════════════════════════
# kailash-ml's ``PreprocessingPipeline.setup()`` fits its imputer, its
# categorical encoder (including TARGET encoding) and its scaler on EVERY
# row it is given, and only afterwards splits train/test. Handing it the
# whole dataset therefore lets test-set statistics leak into training.
#
# The helpers below hold the test rows out FIRST, fit the pipeline on the
# training rows only, and push the held-out rows through the fitted
# ``transform()``. ``setup()`` still splits internally, but that split now
# happens inside the training rows, so it cannot leak anything.

_SETUP_OPTIONS: frozenset[str] = frozenset(
    {
        "normalize",
        "normalize_method",
        "categorical_encoding",
        "imputation_strategy",
        "impute_n_neighbors",
        "remove_outliers",
        "outlier_threshold",
        "max_cardinality",
        "exclude_columns",
        "remove_multicollinearity",
        "multicollinearity_threshold",
        "pca",
        "pca_components",
        "transform_target",
        "fix_imbalance",
        "imbalance_method",
    }
)

# Arguments the helpers own themselves; passing them through would
# re-introduce the leak (``data`` = the full frame) or fight the split.
_OWNED_BY_HELPER: frozenset[str] = frozenset({"data", "target", "train_size", "seed"})


@dataclass(frozen=True)
class PreprocessResult:
    """Outcome of :func:`split_then_preprocess` / :func:`preprocess_train_test`.

    Mirrors the fields of kailash-ml's ``SetupResult`` that exercises use,
    plus the fitted pipeline and a record of what was done to which split.

    ``train_data`` / ``test_data``
        Both produced by the SAME pipeline, fitted on the training rows only.
    ``transformers``
        The fitted imputer / encoder / scaler state (e.g.
        ``transformers["ordinal_mappings"]``), learned from training rows.
    ``train_only_steps``
        Steps applied to the training split only (outlier removal,
        class-imbalance resampling). The test split is never filtered or
        resampled — it stays a faithful sample of the data.
    ``test_null_fills``
        Columns that were complete in the training rows but have gaps in
        the test rows (the fitted pipeline has no learned fill for them),
        mapped to the TRAINING statistic used to fill them.
    """

    train_data: pl.DataFrame
    test_data: pl.DataFrame
    target_column: str
    task_type: str
    numeric_columns: list[str]
    categorical_columns: list[str]
    transformers: dict[str, Any]
    pipeline: Any
    original_shape: tuple[int, int]
    train_raw_rows: int
    test_raw_rows: int
    stratified: bool
    train_only_steps: tuple[str, ...] = ()
    test_null_fills: dict[str, Any] = field(default_factory=dict)
    summary: str = ""


def _looks_like_classification(series: pl.Series) -> bool:
    """Same rule kailash-ml uses to call a target "classification"."""
    if series.dtype in (pl.Boolean, pl.Categorical, pl.Utf8, pl.String):
        return True
    return series.drop_nulls().n_unique() <= 20


def split_raw_train_test(
    df: pl.DataFrame,
    target: str,
    *,
    test_size: float | int = 0.2,
    seed: int = 42,
    stratify: bool | Literal["auto"] = "auto",
) -> tuple[pl.DataFrame, pl.DataFrame, bool]:
    """Hold out test rows from the RAW frame before anything is fitted.

    Rows are shuffled with ``seed``. ``test_size`` is a fraction in (0, 1)
    or an exact row count. With ``stratify="auto"`` a classification target
    (same rule as kailash-ml) is stratified so every class keeps its share
    in both splits; ``True`` / ``False`` force the choice.

    Returns ``(train_raw, test_raw, stratified)``; both splits keep the
    shuffled order.
    """
    if target not in df.columns:
        raise ValueError(f"Target column '{target}' not found in data.")
    n = df.height
    if n < 2:
        raise ValueError(f"Need at least 2 rows to split, got {n}.")
    if isinstance(test_size, np.integer):
        test_size = int(test_size)
    if isinstance(test_size, bool) or not isinstance(test_size, (int, float)):
        raise TypeError(
            f"test_size must be a fraction in (0, 1) or a row count, got {test_size!r}."
        )
    if isinstance(test_size, int):
        if not 1 <= test_size < n:
            raise ValueError(f"test_size as a row count must be in [1, {n - 1}], got {test_size}.")
        test_fraction = test_size / n
    else:
        if not 0.0 < test_size < 1.0:
            raise ValueError(f"test_size as a fraction must be in (0, 1), got {test_size}.")
        test_fraction = float(test_size)

    is_classification = _looks_like_classification(df[target])
    if stratify == "auto":
        stratified = is_classification
    elif stratify is True:
        if not is_classification:
            raise ValueError(
                f"stratify=True needs a classification target; '{target}' has "
                f"{df[target].drop_nulls().n_unique()} distinct values."
            )
        stratified = True
    elif stratify is False:
        stratified = False
    else:
        raise TypeError(f"stratify must be True, False or 'auto', got {stratify!r}.")

    perm = np.random.RandomState(seed).permutation(n)
    shuffled = df[perm.tolist()]

    if stratified:
        # Within each class (in shuffled order), the first round(n_c * f)
        # rows go to test — the class share is the same in both splits.
        is_test = pl.int_range(pl.len()).over(target) < (
            pl.len().over(target) * test_fraction
        ).round()
        flagged = shuffled.with_columns(is_test.alias("__split_is_test"))
        test_raw = flagged.filter(pl.col("__split_is_test")).drop("__split_is_test")
        train_raw = flagged.filter(~pl.col("__split_is_test")).drop("__split_is_test")
    else:
        n_test = test_size if isinstance(test_size, int) else int(round(n * test_fraction))
        train_raw = shuffled.head(n - n_test)
        test_raw = shuffled.tail(n_test)

    if train_raw.height == 0 or test_raw.height == 0:
        raise ValueError(
            f"Split produced an empty side (train={train_raw.height}, "
            f"test={test_raw.height}); adjust test_size."
        )
    return train_raw, test_raw, stratified


def _fill_unfitted_test_nulls(
    test_raw: pl.DataFrame,
    train_raw: pl.DataFrame,
    fitted: Any,
    strategy: str,
) -> tuple[pl.DataFrame, dict[str, Any]]:
    """Fill test gaps the fitted pipeline has no learned value for.

    kailash-ml only stores a fill value for columns that had gaps in the
    rows it was fitted on. A column that is complete in the training rows
    but has gaps in the test rows would otherwise reach the model as null.
    The fill is the TRAINING statistic for the same strategy (never a test
    statistic), and every such column is logged and returned.
    """
    if strategy == "drop":
        return test_raw, {}  # transform() drops incomplete rows itself
    learned = set(fitted.transformers.get("imputer_stats", {}))
    if strategy in ("knn", "iterative"):
        learned |= set(fitted.transformers.get("sklearn_imputer_cols", []))

    fills: dict[str, Any] = {}
    for col in fitted.numeric_columns + fitted.categorical_columns:
        if col in learned or col not in test_raw.columns:
            continue
        if test_raw[col].null_count() == 0:
            continue
        train_col = train_raw[col].drop_nulls()
        if col in fitted.categorical_columns or strategy == "mode":
            modes = train_col.mode().sort()
            if len(modes) == 0:
                raise ValueError(f"Cannot fill '{col}': no observed training values.")
            value = modes[0]
        elif strategy == "median":
            value = train_col.median()
        else:  # mean (and the numeric fallback for any other strategy)
            value = train_col.mean()
        if value is None:
            raise ValueError(f"Cannot fill '{col}': no observed training values.")
        fills[col] = value
    if fills:
        logger.warning(
            "Test rows have gaps in columns that were complete in training; "
            "filled with TRAINING statistics: %s",
            fills,
        )
        test_raw = test_raw.with_columns(
            [pl.col(c).fill_null(v) for c, v in fills.items()]
        )
    return test_raw, fills


def preprocess_train_test(
    train_raw: pl.DataFrame,
    test_raw: pl.DataFrame,
    target: str,
    *,
    seed: int = 42,
    **setup_kwargs: Any,
) -> PreprocessResult:
    """Fit ``PreprocessingPipeline`` on ``train_raw`` only; transform both.

    ``setup_kwargs`` are ``PreprocessingPipeline.setup()`` options
    (``normalize``, ``categorical_encoding``, ``imputation_strategy`` …).
    Steps that change the ROWS of a split are applied to training rows only:

    * ``remove_outliers`` — outlier rows are dropped from training only.
    * ``fix_imbalance`` — SMOTE / ADASYN (or the class-weight flag) is
      applied to the whole transformed training split, never to test.

    Use this directly inside cross-validation folds; use
    :func:`split_then_preprocess` for a single train/test split.
    """
    from kailash_ml import PreprocessingPipeline

    owned = sorted(_OWNED_BY_HELPER & set(setup_kwargs))
    if owned:
        raise TypeError(
            f"{owned} are set by the split-first helper itself and cannot be "
            "passed through to setup()."
        )
    unknown = sorted(set(setup_kwargs) - _SETUP_OPTIONS)
    if unknown:
        raise TypeError(f"Unknown PreprocessingPipeline.setup() options: {unknown}.")
    for name, frame in (("train_raw", train_raw), ("test_raw", test_raw)):
        if target not in frame.columns:
            raise ValueError(f"Target column '{target}' not found in {name}.")
    if train_raw.height == 0 or test_raw.height == 0:
        raise ValueError("train_raw and test_raw must both have rows.")
    if set(train_raw.columns) != set(test_raw.columns):
        raise ValueError("train_raw and test_raw must have the same columns.")

    fix_imbalance = bool(setup_kwargs.pop("fix_imbalance", False))
    imbalance_method = setup_kwargs.get("imbalance_method", "smote")
    remove_outliers = bool(setup_kwargs.get("remove_outliers", False))

    pipeline = PreprocessingPipeline()
    # The pipeline only ever sees training rows. Its internal split stays
    # inside them; resampling is applied below to the full training split.
    fitted = pipeline.setup(
        data=train_raw, target=target, seed=seed, fix_imbalance=False, **setup_kwargs
    )

    train_only: list[str] = []
    if remove_outliers:
        # setup() removed the outlier rows (training rows only); its two
        # internal pieces together are every surviving training row.
        train_data = pl.concat([fitted.train_data, fitted.test_data], how="vertical")
        train_only.append(
            f"remove_outliers: {train_raw.height - train_data.height} training rows dropped"
        )
    else:
        train_data = pipeline.transform(train_raw)

    strategy = setup_kwargs.get("imputation_strategy", "mean")
    test_filled, test_fills = _fill_unfitted_test_nulls(test_raw, train_raw, fitted, strategy)
    test_data = pipeline.transform(test_filled)

    if fix_imbalance and fitted.task_type == "classification":
        resample = getattr(pipeline, "_apply_imbalance_correction", None)
        if resample is None:
            raise RuntimeError(
                "This kailash-ml version has no imbalance-correction step on "
                "PreprocessingPipeline; cannot apply fix_imbalance to the training split."
            )
        before = train_data.height
        train_data = resample(train_data, target, imbalance_method, seed)
        train_only.append(
            f"fix_imbalance ({imbalance_method}): training rows {before} -> {train_data.height}"
        )

    summary = "\n".join(
        [
            f"Task type: {fitted.task_type}",
            f"Split first: {train_raw.height} train rows / {test_raw.height} test rows "
            "(pipeline fitted on the train rows only)",
            f"Numeric features: {len(fitted.numeric_columns)}",
            f"Categorical features: {len(fitted.categorical_columns)}",
            f"Encoding: {setup_kwargs.get('categorical_encoding', 'onehot')}",
            f"Normalization: {'yes' if setup_kwargs.get('normalize', True) else 'no'}",
            f"Imputation: {strategy}",
            f"Train-only steps: {'; '.join(train_only) if train_only else 'none'}",
            f"Transformed train: {train_data.height} rows x {train_data.width} cols",
            f"Transformed test: {test_data.height} rows x {test_data.width} cols",
        ]
    )

    return PreprocessResult(
        train_data=train_data,
        test_data=test_data,
        target_column=target,
        task_type=fitted.task_type,
        numeric_columns=list(fitted.numeric_columns),
        categorical_columns=list(fitted.categorical_columns),
        transformers=dict(fitted.transformers),
        pipeline=pipeline,
        original_shape=(train_raw.height + test_raw.height, train_raw.width),
        train_raw_rows=train_raw.height,
        test_raw_rows=test_raw.height,
        stratified=False,
        train_only_steps=tuple(train_only),
        test_null_fills=test_fills,
        summary=summary,
    )


def split_then_preprocess(
    df: pl.DataFrame,
    target: str,
    *,
    test_size: float | int = 0.2,
    seed: int = 42,
    stratify: bool | Literal["auto"] = "auto",
    **setup_kwargs: Any,
) -> PreprocessResult:
    """Leak-free replacement for ``PreprocessingPipeline().setup(df, ...)``.

    1. Split the RAW frame first (seeded shuffle; stratified on the target
       for classification when ``stratify="auto"``).
    2. Fit ``PreprocessingPipeline`` on the training rows only.
    3. ``transform()`` the test rows with the fitted pipeline.

    ``setup_kwargs`` are passed to ``PreprocessingPipeline.setup()``; see
    :func:`preprocess_train_test` for how row-changing options are handled.
    """
    train_raw, test_raw, stratified = split_raw_train_test(
        df, target, test_size=test_size, seed=seed, stratify=stratify
    )
    result = preprocess_train_test(train_raw, test_raw, target, seed=seed, **setup_kwargs)
    split_line = "stratified on the target" if stratified else "random"
    return replace(
        result,
        stratified=stratified,
        summary=result.summary.replace("Split first:", f"Split first ({split_line}):", 1),
    )


# ── shared/data_loader.py ──
"""Unified data loading for MLFP course — supports local and Colab."""


import logging
from pathlib import Path

import polars as pl

logger = logging.getLogger(__name__)

# Google Drive shared folder containing all MLFP datasets
_DRIVE_FOLDER_ID = "16c3RkGmiwMWbjD7cJKbJx-JRZlgmQdws"

# Module subfolders on the shared Drive
_MODULES = {
    "mlfp01",
    "mlfp02",
    "mlfp03",
    "mlfp04",
    "mlfp05",
    "mlfp06",
    "mlfp_assessment",
}


def _is_colab() -> bool:
    """Detect if running inside Google Colab."""
    try:
        import google.colab  # noqa: F401

        return True
    except ImportError:
        return False


def _colab_data_root() -> Path:
    """Return the Drive-mounted mlfp_data path in Colab."""
    return Path("/content/drive/MyDrive/mlfp_data")


def _local_cache_dir() -> Path:
    """Return local cache directory for downloaded files."""
    cache = Path.cwd() / ".data_cache"
    cache.mkdir(exist_ok=True)
    return cache


def _download_from_drive(module: str, filename: str, dest: Path) -> Path:
    """Download a file from the shared Google Drive using gdown."""
    import gdown

    dest_dir = dest / module
    dest_dir.mkdir(parents=True, exist_ok=True)
    dest_file = dest_dir / filename

    if dest_file.exists():
        logger.debug("Using cached file: %s", dest_file)
        return dest_file

    # gdown can download from a folder by file path
    url = f"https://drive.google.com/drive/folders/{_DRIVE_FOLDER_ID}"
    logger.info("Downloading %s/%s from Google Drive...", module, filename)

    # Download the specific file from the shared folder
    try:
        gdown.download_folder(
            url=url,
            output=str(dest),
            quiet=True,
            remaining_ok=True,
        )
    except TypeError:
        # Older gdown versions don't support remaining_ok
        gdown.download_folder(
            url=url,
            output=str(dest),
            quiet=True,
        )

    if not dest_file.exists():
        # Try direct download if folder download didn't isolate the file
        for candidate in dest.rglob(filename):
            if candidate.is_file():
                if candidate != dest_file:
                    candidate.rename(dest_file)
                return dest_file

        msg = (
            f"File not found after download: {module}/{filename}. "
            f"Check that it exists in the mlfp_data shared Drive."
        )
        raise FileNotFoundError(msg)

    return dest_file


def _read_file(path: Path) -> pl.DataFrame:
    """Read a data file into a polars DataFrame."""
    suffix = path.suffix.lower()
    if suffix == ".csv":
        return pl.read_csv(path, try_parse_dates=True)
    elif suffix == ".parquet":
        return pl.read_parquet(path)
    elif suffix == ".json":
        return pl.read_json(path)
    elif suffix in (".p", ".pickle", ".pkl"):
        import pickle

        with open(path, "rb") as f:
            obj = pickle.load(f)  # noqa: S301
        if isinstance(obj, pl.DataFrame):
            return obj
        raise TypeError(
            f"Cannot convert pickle object of type {type(obj)} to polars DataFrame. "
            f"Convert the pickle to parquet upstream: pl.from_pandas(obj).write_parquet('out.parquet')"
        )
    else:
        raise ValueError(
            f"Unsupported file format: {suffix}. Use .csv, .parquet, or .json"
        )


def _repo_data_dir() -> Path | None:
    """Find the repo-local data/ directory by walking up from cwd."""
    for parent in [Path.cwd(), *Path.cwd().parents]:
        candidate = parent / "data"
        if candidate.is_dir() and (parent / "pyproject.toml").exists():
            return candidate
    return None


class MLFPDataLoader:
    """Load MLFP course datasets with automatic source resolution.

    Resolution order:
    1. Colab: Drive mount at /content/drive/MyDrive/mlfp_data/
    2. Local repo data/ directory (committed datasets)
    3. Google Drive download via gdown (cached in .data_cache/)

    Usage:
        loader = MLFPDataLoader()
        df = loader.load("mlfp01", "hdbprices.csv")

    Shortcut:
        df = MLFPDataLoader.mlfp01("hdbprices.csv")
    """

    def __init__(self, cache_dir: Path | str | None = None):
        self._colab = _is_colab()
        if self._colab:
            self._root = _colab_data_root()
        else:
            self._local_data = _repo_data_dir()
            self._cache = Path(cache_dir) if cache_dir else _local_cache_dir()

    def load_raw(self, module: str, filename: str) -> Path:
        """Return the file path without reading into memory.

        Use this for image directories, audio files, or any data that torch/HF
        loads directly rather than via polars.

        Args:
            module: Module subfolder (e.g., "mlfp05")
            filename: File or directory name (e.g., "fashion_mnist", "cifar10")

        Returns:
            Path to the local file or directory.
        """
        if module not in _MODULES:
            raise ValueError(
                f"Unknown module '{module}'. Available: {sorted(_MODULES)}"
            )

        if self._colab:
            path = self._root / module / filename
        else:
            if self._local_data:
                local_path = self._local_data / module / filename
                if local_path.exists():
                    return local_path
            path = self._cache / module / filename

        if not path.exists():
            raise FileNotFoundError(
                f"Raw data not found: {module}/{filename}. "
                f"Run 'python scripts/fetch-real-data.py' to download."
            )
        return path

    @staticmethod
    def load_hf(
        dataset_name: str,
        split: str = "train",
        streaming: bool = False,
    ):
        """Load a HuggingFace dataset directly (not via polars).

        Use this for large datasets (millions of rows) or multimodal data
        (images, audio) that don't fit into a DataFrame.

        Args:
            dataset_name: HuggingFace dataset ID (e.g., "zalando-datasets/fashion_mnist")
            split: Dataset split ("train", "test", "validation")
            streaming: If True, returns an IterableDataset for memory-efficient processing

        Returns:
            HuggingFace Dataset or IterableDataset object.
        """
        from datasets import load_dataset

        logger.info(
            "Loading HuggingFace dataset: %s (split=%s, streaming=%s)",
            dataset_name,
            split,
            streaming,
        )
        return load_dataset(dataset_name, split=split, streaming=streaming)

    def load(self, module: str, filename: str) -> pl.DataFrame:
        """Load a dataset file as a polars DataFrame.

        Args:
            module: Module subfolder (e.g., "mlfp01", "mlfp_assessment")
            filename: File name within the module folder (e.g., "hdbprices.csv")

        Returns:
            polars DataFrame with the loaded data.
        """
        if module not in _MODULES:
            raise ValueError(
                f"Unknown module '{module}'. Available: {sorted(_MODULES)}"
            )

        if self._colab:
            path = self._root / module / filename
            if not path.exists():
                raise FileNotFoundError(
                    f"File not found: {path}. "
                    f"Ensure mlfp_data is accessible in your Google Drive."
                )
        else:
            # Check repo-local data/ first, then fall back to Drive download
            if self._local_data:
                module_dir = self._local_data / module
                local_path = module_dir / filename
                if local_path.exists():
                    path = local_path
                    logger.info(
                        "Loading %s/%s from local data/ (%s)", module, filename, path
                    )
                    return _read_file(path)
                # Bundled data is present for this module but the file is absent
                # → it is genuinely missing. Raise immediately instead of falling
                # back to a Drive download that would hang offline (and pulls the
                # whole folder online just to discover the file is not there).
                if module_dir.is_dir():
                    raise FileNotFoundError(
                        f"File not found: {module}/{filename} "
                        f"(not in local data/{module}/)."
                    )
            path = _download_from_drive(module, filename, self._cache)

        logger.info("Loading %s/%s (%s)", module, filename, path)
        return _read_file(path)

    def list_files(self, module: str) -> list[str]:
        """List available data files in a module folder."""
        if module not in _MODULES:
            raise ValueError(
                f"Unknown module '{module}'. Available: {sorted(_MODULES)}"
            )

        if self._colab:
            root = self._root / module
        else:
            root = self._cache / module

        if not root.exists():
            return []

        return sorted(f.name for f in root.iterdir() if f.is_file())

    # -- Module shortcuts --

    @classmethod
    def mlfp01(cls, filename: str) -> pl.DataFrame:
        """Load from mlfp01 (Data Pipelines & Visualisation)."""
        return cls().load("mlfp01", filename)

    @classmethod
    def mlfp02(cls, filename: str) -> pl.DataFrame:
        """Load from mlfp02 (Statistical Mastery)."""
        return cls().load("mlfp02", filename)

    @classmethod
    def mlfp03(cls, filename: str) -> pl.DataFrame:
        """Load from mlfp03 (Supervised ML)."""
        return cls().load("mlfp03", filename)

    @classmethod
    def mlfp04(cls, filename: str) -> pl.DataFrame:
        """Load from mlfp04 (Unsupervised ML)."""
        return cls().load("mlfp04", filename)

    @classmethod
    def mlfp05(cls, filename: str) -> pl.DataFrame:
        """Load from mlfp05 (Deep Learning & Vision)."""
        return cls().load("mlfp05", filename)

    @classmethod
    def mlfp06(cls, filename: str) -> pl.DataFrame:
        """Load from mlfp06 (LLMs, Agents & Transformation)."""
        return cls().load("mlfp06", filename)

    @classmethod
    def assessment(cls, filename: str) -> pl.DataFrame:
        """Load from mlfp_assessment (capstone datasets)."""
        return cls().load("mlfp_assessment", filename)


# ── shared/mlfp02/ex_3.py ──
"""
Shared infrastructure for MLFP02 Exercise 3 — A/B testing & multiple comparisons.

Contains: experiment data loading, group extraction, SRM check, common constants,
and small statistical helpers reused across the four technique files:

    01_bootstrap_power.py    — bootstrap CIs + MDE + power curves
    02_hypothesis_testing.py — two-proportion z-test + effect sizes
    03_multiple_testing.py   — Bonferroni + BH-FDR + FDR simulation
    04_permutation_test.py   — distribution-free alternative
    05_parametric_bootstrap.py — resample from a fitted model vs the data
    06_one_sample_one_tailed.py — one-sample t + directional tests

Technique-specific code (the actual corrections, permutation loops, power
formulas) does NOT belong here — each technique file owns its own logic.
"""

from pathlib import Path
from typing import Any, Callable

import numpy as np
import polars as pl
from scipy import stats


# ════════════════════════════════════════════════════════════════════════
# CONSTANTS
# ════════════════════════════════════════════════════════════════════════

ALPHA: float = 0.05
POWER_TARGET: float = 0.80
N_BOOTSTRAP: int = 10_000
N_PERMUTATIONS: int = 10_000
RANDOM_SEED: int = 42

OUTPUT_DIR = Path("outputs") / "mlfp02_ex3_ab_testing"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# The experiment was DESIGNED with unequal allocation across four arms.
# SRM must be tested against this design, never against a default 50/50.
DESIGNED_ALLOCATION: dict[str, float] = {
    "control": 0.40,
    "treatment_a": 0.35,
    "treatment_b": 0.15,
    "variant_c": 0.10,
}

# Exercise 3 compares ONE treatment arm with control. Pooling several
# different treatments into one "treatment" group would estimate a
# meaningless mixture of effects.
TREATMENT_ARM: str = "treatment_a"

# Binary success event: a "qualifying order" — a basket of at least $50
# (metric_value is basket value in SGD). Nearly every user has
# metric_value > 0, so "> 0" would give a ~99% "conversion" rate.
CONVERSION_THRESHOLD: float = 50.0


# ════════════════════════════════════════════════════════════════════════
# DATA LOADING — Singapore e-commerce A/B test
# ════════════════════════════════════════════════════════════════════════


def load_experiment_all() -> pl.DataFrame:
    """Load every arm of the experiment (control + three variants).

    Columns: user_id, experiment_group, metric_value, pre_metric_value,
             revenue, timestamp, segment, platform, country, converted.

    `converted` = 1 when the user placed a qualifying order
    (metric_value >= CONVERSION_THRESHOLD).
    """
    loader = MLFPDataLoader()
    df = loader.load("mlfp02", "experiment_data.parquet")
    return df.with_columns(
        (pl.col("metric_value") >= CONVERSION_THRESHOLD)
        .cast(pl.Int8)
        .alias("converted")
    )


def designed_control_share(arm: str = TREATMENT_ARM) -> float:
    """Designed share of control within the (control, arm) pair."""
    c = DESIGNED_ALLOCATION["control"]
    return c / (c + DESIGNED_ALLOCATION[arm])


def load_experiment(arm: str = TREATMENT_ARM) -> pl.DataFrame:
    """Control vs ONE treatment arm, with a `group` column in {control, treatment}.

    Only these two arms are kept, so every downstream comparison is a
    clean two-arm contrast. Run `srm_check_multi` on `load_experiment_all()`
    first to confirm the arms you analyse were allocated as designed.
    """
    df = load_experiment_all().filter(
        pl.col("experiment_group").is_in(["control", arm])
    )
    return df.with_columns(
        pl.when(pl.col("experiment_group") == "control")
        .then(pl.lit("control"))
        .otherwise(pl.lit("treatment"))
        .alias("group")
    )


def split_groups(df: pl.DataFrame) -> tuple[pl.DataFrame, pl.DataFrame]:
    """Return (control_df, treatment_df)."""
    control = df.filter(pl.col("group") == "control")
    treatment = df.filter(pl.col("group") == "treatment")
    return control, treatment


def conversion_arrays(df: pl.DataFrame) -> tuple[np.ndarray, np.ndarray]:
    """Return (control_converted, treatment_converted) as float64 arrays."""
    control, treatment = split_groups(df)
    c = control["converted"].to_numpy().astype(np.float64)
    t = treatment["converted"].to_numpy().astype(np.float64)
    return c, t


def revenue_arrays(df: pl.DataFrame) -> tuple[np.ndarray, np.ndarray]:
    """Return (control_revenue, treatment_revenue) as float64 arrays."""
    control, treatment = split_groups(df)
    c = control["revenue"].to_numpy().astype(np.float64)
    t = treatment["revenue"].to_numpy().astype(np.float64)
    return c, t


# ════════════════════════════════════════════════════════════════════════
# SANITY CHECKS — SRM
# ════════════════════════════════════════════════════════════════════════


def srm_check(
    n_control: int, n_treatment: int, expected_ratio: float
) -> dict[str, Any]:
    """χ² goodness-of-fit test for Sample Ratio Mismatch on a two-arm pair.

    `expected_ratio` is the DESIGNED share of control within the pair
    (see `designed_control_share`). There is deliberately no 50/50
    default: testing an unequal design against 50/50 always "detects" SRM.

    Returns dict with chi2, p_value, a boolean `srm`, and a verdict.
    SRM indicates randomisation bugs, bot traffic, or pipeline issues —
    if p < 0.01 do NOT trust downstream test results.
    """
    n_total = n_control + n_treatment
    expected = np.array([n_total * expected_ratio, n_total * (1 - expected_ratio)])
    observed = np.array([n_control, n_treatment])
    chi2, p = stats.chisquare(observed, f_exp=expected)
    verdict = (
        "SRM DETECTED — investigate randomisation"
        if p < 0.01
        else "OK — sample split consistent"
    )
    return {"chi2": float(chi2), "p_value": float(p), "srm": bool(p < 0.01), "verdict": verdict}


def srm_check_multi(
    counts: dict[str, int], allocation: dict[str, float] = DESIGNED_ALLOCATION
) -> dict[str, Any]:
    """χ² SRM test across ALL arms against the designed allocation.

    Returns chi2, p_value, srm flag, and a per-arm table (observed share,
    designed share, standardised residual) so you can see WHICH arm is
    mis-allocated, not just that something is wrong.
    """
    arms = list(allocation)
    observed = np.array([counts[a] for a in arms], dtype=np.float64)
    n_total = observed.sum()
    expected = np.array([allocation[a] for a in arms]) * n_total
    chi2, p = stats.chisquare(observed, f_exp=expected)
    per_arm = {
        a: {
            "observed": int(observed[i]),
            "observed_share": float(observed[i] / n_total),
            "designed_share": float(allocation[a]),
            "std_residual": float((observed[i] - expected[i]) / np.sqrt(expected[i])),
        }
        for i, a in enumerate(arms)
    }
    return {
        "chi2": float(chi2),
        "p_value": float(p),
        "srm": bool(p < 0.01),
        "per_arm": per_arm,
    }


# ════════════════════════════════════════════════════════════════════════
# SMALL REUSABLE STATS
# ════════════════════════════════════════════════════════════════════════


def two_proportion_ztest(
    p_control: float, p_treatment: float, n_control: int, n_treatment: int
) -> tuple[float, float]:
    """Pooled two-proportion z-test. Returns (z_stat, two_sided_p_value)."""
    p_pool = (p_control * n_control + p_treatment * n_treatment) / (
        n_control + n_treatment
    )
    se = np.sqrt(p_pool * (1 - p_pool) * (1 / n_control + 1 / n_treatment))
    z = (p_treatment - p_control) / se if se > 0 else 0.0
    p_value = 2 * (1 - stats.norm.cdf(abs(z)))
    return float(z), float(p_value)


def cohens_h(p1: float, p2: float) -> float:
    """Cohen's h effect size for two proportions."""
    return float(2 * (np.arcsin(np.sqrt(p2)) - np.arcsin(np.sqrt(p1))))


def cohens_d(x1: np.ndarray, x2: np.ndarray) -> float:
    """Pooled Cohen's d effect size for two samples."""
    s_pool = np.sqrt((x1.var(ddof=1) + x2.var(ddof=1)) / 2)
    return float((x2.mean() - x1.mean()) / s_pool) if s_pool > 0 else 0.0


def interpret_magnitude(abs_effect: float) -> str:
    """Cohen convention: <0.2 negligible, <0.5 small, <0.8 medium, else large."""
    if abs_effect < 0.2:
        return "negligible"
    if abs_effect < 0.5:
        return "small"
    if abs_effect < 0.8:
        return "medium"
    return "large"


def print_header(title: str) -> None:
    """Consistent banner for each technique file."""
    print("=" * 70)
    print(f"  {title}")


# ════════════════════════════════════════════════════════════════════════
# PARAMETRIC BOOTSTRAP — resample from a FITTED model, not the data
# ════════════════════════════════════════════════════════════════════════


def parametric_bootstrap_statistic(
    sampler: Callable[[np.random.Generator, int], np.ndarray],
    n: int,
    statistic: Callable[[np.ndarray], float],
    n_boot: int = N_BOOTSTRAP,
    seed: int = RANDOM_SEED,
) -> np.ndarray:
    """Parametric bootstrap: draw fresh size-n samples from a fitted model.

    ``sampler(rng, n)`` must return n synthetic draws from the fitted
    distribution (e.g. a Normal parameterised by the sample moments). The
    bootstrap distribution of ``statistic`` is then computed entirely under
    the model — powerful when the model is right, misleading when it is not.
    """
    rng = np.random.default_rng(seed)
    out = np.empty(n_boot, dtype=np.float64)
    for i in range(n_boot):
        out[i] = statistic(sampler(rng, n))
    return out


# ════════════════════════════════════════════════════════════════════════
# EXPERIMENT TRACKING — kailash-ml ExperimentTracker
# ════════════════════════════════════════════════════════════════════════

TRACKER_STORE_URL = (
    f"sqlite:///{(OUTPUT_DIR / 'experiments.db').resolve().as_posix()}"
)


def track_train_run(
    experiment: str,
    run_name: str,
    params: dict[str, str],
    metrics: dict[str, float],
) -> str:
    """Log one Train-phase run to ExperimentTracker (sync wrapper).

    Returns the run_id. The tracker is closed in a finally block — kailash-ml
    holds the store connection open until close() is called.
    """
    import asyncio

    async def _log() -> str:
        from kailash_ml import ExperimentTracker

        tracker = await ExperimentTracker.create(store_url=TRACKER_STORE_URL)
        try:
            async with tracker.track(
                experiment=experiment, run_name=run_name
            ) as run:
                await run.log_params(params)
                await run.log_metrics(metrics)
                return run.run_id
        finally:
            await tracker.close()

    return asyncio.run(_log())
    print("=" * 70)




# ════════════════════════════════════════════════════════════════════════
# MLFP02 — Exercise 3.4: Permutation Test
# ════════════════════════════════════════════════════════════════════════
#
# WHAT YOU'LL LEARN:
#   - Implement a permutation test from scratch
#   - Understand why permutation tests require no distributional assumptions
#   - Compare permutation p-values with parametric p-values
#   - Apply permutation tests to both conversion and revenue metrics
#   - Visualise the permutation null distribution
#
# PREREQUISITES: Exercise 3.2 (hypothesis testing)
# ESTIMATED TIME: ~35 min
#
# TASKS:
#   1. Load data and compute observed differences
#   2. Permutation test for conversion rate (10K permutations)
#   3. Permutation test for revenue (non-Normal metric)
#   4. Compare with parametric results
#   5. Visualise permutation null distributions
#
# THEORY:
#   Permutation test algorithm:
#     1. Pool ALL observations (control + treatment)
#     2. Randomly assign to "control" and "treatment" (preserving sizes)
#     3. Compute test statistic on permuted data
#     4. Repeat 10K times -> null distribution
#     5. p-value = proportion of |permuted stat| >= |observed stat|
#
#   Key advantage: NO assumptions about the shape of the distribution.
#   If the parametric test assumes Normality and the data is skewed,
#   the permutation test gives a more trustworthy p-value.
#
# ════════════════════════════════════════════════════════════════════════


In [ ]:
from __future__ import annotations

import numpy as np
import plotly.graph_objects as go
import polars as pl
from plotly.subplots import make_subplots
from scipy import stats


print_header("MLFP02 Exercise 3.4: Permutation Test")



## TASK 1 — Load data and compute observed differences


In [ ]:
df = load_experiment()
control, treatment = split_groups(df)
n_control = control.height
n_treatment = treatment.height

ctrl_conv, treat_conv = conversion_arrays(df)
ctrl_rev, treat_rev = revenue_arrays(df)

observed_conv_diff = treat_conv.mean() - ctrl_conv.mean()
observed_rev_diff = treat_rev.mean() - ctrl_rev.mean()

print(f"Data loaded: {df.height:,} users")
print(f"  Observed conversion diff: {observed_conv_diff:+.6f}")
print(f"  Observed revenue diff:    ${observed_rev_diff:+.2f}")



### Checkpoint 1


In [ ]:
assert n_control > 0 and n_treatment > 0, "Both groups must have data"
print("\n>>> Checkpoint 1 passed -- data loaded\n")



## THEORY — Why permutation tests?


In [ ]:
# The z-test assumes the sampling distribution of the difference is
# Normal. For conversion rates with large n, the CLT guarantees this.
# But for revenue (highly skewed, heavy-tailed), the CLT approximation
# may be poor even at n=5000.
#
# The permutation test asks a simpler question: "If the treatment had
# NO effect, how likely is it that we'd see a difference this large
# just by random assignment?"
#
# We answer this by ACTUALLY doing the random assignment 10,000 times:
#   - Pool all users together (erasing group labels)
#   - Randomly assign n_control to "control", rest to "treatment"
#   - Compute the difference in the permuted groups
#   - Count how often the permuted difference is as extreme as observed
#
# Analogy: Imagine you're told a coin is biased. You flip it 100 times
# and get 58 heads. Is that enough evidence? You could compute the
# binomial test (parametric). OR you could shuffle a deck of 100 cards
# (58 red, 42 black), deal them into two piles, and count how often
# the difference is as extreme as 58 vs 42. No math needed -- just
# repeated shuffling.



## TASK 2 — Permutation test for conversion rate


In [ ]:
rng = np.random.default_rng(seed=RANDOM_SEED)
all_converted = df["converted"].to_numpy()

perm_conv_diffs = np.zeros(N_PERMUTATIONS)
for i in range(N_PERMUTATIONS):
    # TODO: Randomly permute all_converted using rng.permutation.
    # Then split into first n_control (permuted control) and rest (permuted treatment).
    # Compute the difference: perm_treat_rate - perm_ctrl_rate.
    # Hint: perm = rng.permutation(all_converted)
    #        perm_ctrl_rate = perm[:n_control].mean()
    #        perm_treat_rate = perm[n_control:].mean()
    perm = ____
    perm_ctrl_rate = ____
    perm_treat_rate = ____
    perm_conv_diffs[i] = perm_treat_rate - perm_ctrl_rate

# TODO: Compute the permutation p-value with the add-one correction:
# (number of |perm_diffs| >= |observed| + 1) / (N_PERMUTATIONS + 1).
# The observed labelling is itself one valid permutation, so p is never 0.
# Hint: count with np.sum(np.abs(perm_conv_diffs) >= np.abs(observed_conv_diff))
n_extreme_conv = ____
perm_p_conversion = ____

print(f"=== Permutation Test (conversion rate) ===")
print(f"Observed difference: {observed_conv_diff:+.6f}")
print(
    f"Permutation null: mean={perm_conv_diffs.mean():.6f}, "
    f"std={perm_conv_diffs.std():.6f}"
)
print(
    f"Permutation p-value: {perm_p_conversion:.6f} "
    f"({n_extreme_conv} of {N_PERMUTATIONS:,} permutations as extreme)"
)

# Compare with parametric
_, parametric_p = two_proportion_ztest(
    ctrl_conv.mean(), treat_conv.mean(), n_control, n_treatment
)
print(f"z-test p-value:      {parametric_p:.6f}")
agreement = (perm_p_conversion < ALPHA) == (parametric_p < ALPHA)
print(f"Agreement: {'YES' if agreement else 'NO'}")



### Checkpoint 2


In [ ]:
assert 0 < perm_p_conversion <= 1, "Permutation p-value must be in (0, 1]"
assert len(perm_conv_diffs) == N_PERMUTATIONS, "Should have N_PERMUTATIONS samples"
assert (
    abs(perm_conv_diffs.mean()) < 0.01
), "Permutation null should be centred near zero"
print("\n>>> Checkpoint 2 passed -- conversion permutation test completed\n")



## TASK 3 — Permutation test for revenue (non-Normal)


In [ ]:
# Revenue is typically right-skewed (many small purchases, few large).
# The permutation test below uses the DIFFERENCE IN MEANS, so its
# like-for-like parametric comparison is Welch's t-test (also a mean
# difference, relying on the CLT). Mann-Whitney U is a different,
# NON-parametric test: it asks whether one group's values tend to be
# larger (stochastic ordering), not whether the means differ.

all_revenue = df["revenue"].to_numpy().astype(np.float64)

perm_rev_diffs = np.zeros(N_PERMUTATIONS)
for i in range(N_PERMUTATIONS):
    # TODO: Same procedure as above but for revenue.
    # Hint: perm = rng.permutation(all_revenue)
    #        perm_rev_diffs[i] = perm[n_control:].mean() - perm[:n_control].mean()
    perm = ____
    perm_rev_diffs[i] = ____

# TODO: Compute the add-one permutation p-value for revenue.
n_extreme_rev = ____
perm_p_revenue = ____

print(f"=== Permutation Test (revenue) ===")
print(f"Observed revenue diff: ${observed_rev_diff:+.2f}")
print(
    f"Permutation null: mean=${perm_rev_diffs.mean():.2f}, "
    f"std=${perm_rev_diffs.std():.2f}"
)
print(
    f"Permutation p-value: {perm_p_revenue:.6f} "
    f"({n_extreme_rev} of {N_PERMUTATIONS:,} permutations as extreme)"
)

# Like-for-like parametric comparison: Welch's t-test on the mean difference
_, welch_p = stats.ttest_ind(treat_rev, ctrl_rev, equal_var=False)
print(f"Welch's t p-value:   {welch_p:.6f}")
rev_agreement = (perm_p_revenue < ALPHA) == (welch_p < ALPHA)
# For reference only — a different hypothesis (stochastic ordering)
_, mwu_p = stats.mannwhitneyu(treat_rev, ctrl_rev, alternative="two-sided")
print(f"Mann-Whitney U p-value (non-parametric, ordering not means): {mwu_p:.6f}")
print(f"Agreement: {'YES' if rev_agreement else 'NO'}")



### Checkpoint 3


In [ ]:
assert 0 < perm_p_revenue <= 1, "Revenue permutation p-value must be in (0, 1]"
assert len(perm_rev_diffs) == N_PERMUTATIONS, "Should have N_PERMUTATIONS samples"
print("\n>>> Checkpoint 3 passed -- revenue permutation test completed\n")



## TASK 4 — Comparison summary


In [ ]:
print(f"=== Parametric vs Permutation Comparison ===")
print(
    f"{'Metric':<20} {'Parametric test':<16} {'Parametric p':>14} "
    f"{'Permutation p':>15} {'Agree':>8}"
)
print("-" * 78)
print(
    f"{'Conversion':<20} {'z-test':<16} {parametric_p:>14.6f} {perm_p_conversion:>15.6f} "
    f"{'YES' if agreement else 'NO':>8}"
)
print(
    f"{'Revenue (mean)':<20} {'Welch t':<16} {welch_p:>14.6f} {perm_p_revenue:>15.6f} "
    f"{'YES' if rev_agreement else 'NO':>8}"
)

# INTERPRETATION: When parametric and permutation tests agree, both
# assumptions hold and you can trust either. When they disagree, the
# permutation test is more trustworthy because it makes fewer
# assumptions. In practice, for conversion rates (binary, large n),
# they almost always agree. For revenue (skewed, heavy-tailed),
# disagreement is more common.



### Checkpoint 4


In [ ]:
print("\n>>> Checkpoint 4 passed -- comparison complete\n")



## TASK 5 — Visualise permutation null distributions


In [ ]:
from kailash_ml import ModelVisualizer

viz = ModelVisualizer()

# Conversion permutation null
conv_df = pl.DataFrame({"permuted_difference": perm_conv_diffs})
fig1 = viz.histogram(
    conv_df,
    column="permuted_difference",
    bins=50,
    title="Permutation Null: Conversion Rate Difference",
)
fig1.add_vline(
    x=observed_conv_diff,
    line_dash="dash",
    annotation_text=f"Observed ({observed_conv_diff:+.4f})",
)
fig1.add_vline(
    x=-abs(observed_conv_diff),
    line_dash="dash",
    line_color="red",
    annotation_text="Mirror",
)
out_1 = OUTPUT_DIR / "permutation_null_conversion.html"
fig1.write_html(str(out_1))
print(f"Saved: {out_1}")

# Revenue permutation null
rev_df = pl.DataFrame({"permuted_revenue_difference": perm_rev_diffs})
fig2 = viz.histogram(
    rev_df,
    column="permuted_revenue_difference",
    bins=50,
    title="Permutation Null: Revenue Difference",
)
fig2.add_vline(
    x=observed_rev_diff,
    line_dash="dash",
    annotation_text=f"Observed (${observed_rev_diff:+.2f})",
)
fig2.add_vline(
    x=-abs(observed_rev_diff),
    line_dash="dash",
    line_color="red",
    annotation_text="Mirror",
)
out_2 = OUTPUT_DIR / "permutation_null_revenue.html"
fig2.write_html(str(out_2))
print(f"Saved: {out_2}")



## APPLY — Distribution-free testing for a Singapore ride-hailing platform


In [ ]:
# SCENARIO: A Singapore ride-hailing platform tests a new surge pricing
# algorithm. The key metric is revenue per ride, which is highly skewed:
# most rides cost S$8-15, but airport rides, peak-hour rides, and
# luxury rides create a heavy right tail (S$50-200).
#
# A t-test on this data is questionable -- the distribution is nowhere
# near Normal, and even with n=10,000 the CLT approximation for the
# mean is unreliable because the variance is dominated by the tail.
#
# The permutation test handles this naturally:
#   - No distributional assumptions
#   - Exact p-value (up to permutation count)
#   - Works for ANY test statistic (mean, median, trimmed mean, ratio)
#
# BUSINESS IMPACT: Without the permutation test, the team would have
# shipped a pricing algorithm that appeared to increase revenue but
# actually just happened to catch a few high-value airport rides in the
# treatment group. Avoiding this false positive saves ~S$200K in
# engineering effort and prevents a price increase that would have
# reduced rider satisfaction with no real revenue benefit.

print(f"\n--- Business Application: Distribution-Free Testing ---")
print("Revenue distributions are rarely Normal.")
print("Permutation tests give reliable p-values regardless of distribution shape.")
print("When parametric and permutation tests disagree, trust the permutation test.")



## REFLECTION


[x] Permutation test: distribution-free, no parametric assumptions
  [x] Permutation null distribution: what "no effect" actually looks like
  [x] Conversion + revenue permutation tests from scratch
  [x] Parametric vs permutation comparison: when they agree and disagree

  EXERCISE 3 COMPLETE: You now have the full A/B testing toolkit:
    3.1: Bootstrap CIs + power analysis (how precise, how sensitive)
    3.2: Hypothesis testing + effect sizes (is it real, is it big enough)
    3.3: Multiple testing corrections (controlling false discoveries)
    3.4: Permutation tests (no assumptions, trustworthy p-values)

  NEXT: In Exercise 4 you'll design a complete experiment from scratch --
  writing hypotheses before data, computing required sample sizes,
  and creating a structured data collection plan.


In [ ]:
print("\n" + "=" * 70)
print("  WHAT YOU'VE MASTERED")
print("=" * 70)
print(
)

print(">>> Exercise 3.4 complete -- Permutation Test")

